# SUPPORT2: repeated-split stability and fixed-group calibration uncertainty

**Corrected calibration sensitivity extension (7 November 2026).**

This version preserves the original eight-predictor Cox, stratified Cox and Random Survival Forest (RSF) model specifications and the ten stratified 80/20 splits. It replaces the *re-created deciles within every bootstrap resample* with **model-specific risk-group cutoffs fixed using the original seed-42 test predictions**. The same patient indices are resampled for all three models (2,000 resamples).

**Interpretation:** Fixed groups target uncertainty of the *same prespecified-on-this-test-set grouped-calibration statistic*, conditional on the fitted models and the original test-set cutoffs. This remains internal, conditional sensitivity analysis, not external validation; choosing cutoffs on the test set and using absolute errors can shift bootstrap distributions. Percentile intervals need not contain original estimates and must not be described as unbiased. Paired difference intervals including zero do not establish model equivalence.

**Execution:** In Google Colab, use **Runtime → Run all**; the final cell creates `support2_new_results_fixed_groups.zip`. Required packages and original dataset are fetched in the notebook; internet is required. Previous bootstrap output files should not be combined with the new fixed-group tables because they estimate different grouped statistics.


In [ ]:
%pip -q install ucimlrepo lifelines scikit-survival statsmodels
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from lifelines import CoxPHFitter
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from ucimlrepo import fetch_ucirepo
import importlib.metadata as versions
OUT=Path('support2_new_results');OUT.mkdir(exist_ok=True)
print({x:versions.version(x) for x in ['lifelines','scikit-survival','scikit-learn','ucimlrepo']})

## Recover source data and validate survival endpoint

Outcome: death observed within 180 days from SUPPORT study entry. A binary ROC-AUC/Brier approach is used **only if early censoring is absent**. Survival follow-up `d.time` must be retrieved from the original dataset rather than from UCI's feature-only frame.


In [ ]:
support2=fetch_ucirepo(id=880)
X,y=support2.data.features,support2.data.targets
df=pd.concat([X,y],axis=1)
df=df.loc[:,~df.columns.duplicated()].copy()
original=support2.data.original
assert df.index.equals(original.index)
assert np.array_equal(df['death'].to_numpy(),original['death'].to_numpy())
df['d.time']=original['d.time'].copy()
assert len(df)==9105
assert df[['death','d.time']].notna().all().all()
assert df['death'].isin([0,1]).all() and (df['d.time']>0).all()
features=['age','sex','dzgroup','num.co','diabetes','dementia','ca','hday']
baseline=df[features].copy()
baseline['time_180']=np.minimum(df['d.time'].astype(float),180.)
baseline['event_180']=((df['death']==1)&(df['d.time']<=180)).astype(int)
assert not baseline.isna().any().any()
early=int(((baseline.event_180==0)&(baseline.time_180<180)).sum())
assert early==0, f'Early censoring ({early}) present; do not use binary metrics without adjustment.'
print('Patients',len(baseline),'180-day deaths',baseline.event_180.sum(),'Early censored',early)

## Model fitting: align exact model families with original notebook

Original Cox `penalizer=0.01` and dummy-coded sex, disease group and cancer status; stratified Cox `strata=['dzgroup']` with dummy-coded sex and cancer status; RSF defaults to `splitter='best'` tree splitting with the survival-specific **log-rank** criterion, 100 trees, `min_samples_split=20`, `min_samples_leaf=20`, `max_features='sqrt'`, `oob_score=True`, `random_state=42`. No systematic hyperparameter tuning. To ensure comparable feature encodings, category levels are established on the training data in each split. For the repeated-split check, the RSF random seed is held at 42 to isolate partition variability.


In [ ]:
def fit_predict(train,test):
    time='time_180'; event='event_180'
    dummy=['sex','dzgroup','ca']
    tr=pd.get_dummies(train,columns=dummy,drop_first=True,dtype=int)
    te=pd.get_dummies(test,columns=dummy,drop_first=True,dtype=int).reindex(columns=tr.columns,fill_value=0)
    constant=[c for c in tr.columns if c not in [time,event] and tr[c].nunique()<=1]
    tr=tr.drop(columns=constant);te=te.reindex(columns=tr.columns,fill_value=0)
    cph=CoxPHFitter(penalizer=0.01)
    cph.fit(tr,duration_col=time,event_col=event)
    risk_cox=1-cph.predict_survival_function(te,times=[180]).iloc[0].reindex(test.index)
    # Stratified Cox: columns sex and ca only encoded, dzgroup remains a stratum variable.
    tr_s=pd.get_dummies(train,columns=['sex','ca'],drop_first=True,dtype=int)
    te_s=pd.get_dummies(test,columns=['sex','ca'],drop_first=True,dtype=int).reindex(columns=tr_s.columns,fill_value=0)
    sc=CoxPHFitter(penalizer=0.01)
    sc.fit(tr_s,duration_col=time,event_col=event,strata=['dzgroup'])
    risk_strat=1-sc.predict_survival_function(te_s,times=[180]).iloc[0].reindex(test.index)
    X_tr=pd.get_dummies(train[features],columns=dummy,dtype=float)
    X_te=pd.get_dummies(test[features],columns=dummy,dtype=float).reindex(columns=X_tr.columns,fill_value=0)
    y_tr=Surv.from_arrays(train[event].astype(bool).to_numpy(),train[time].astype(float).to_numpy())
    rsf=RandomSurvivalForest(n_estimators=100,min_samples_split=20,
      min_samples_leaf=20,max_features='sqrt',oob_score=True,n_jobs=-1,random_state=42)
    rsf.fit(X_tr,y_tr)
    pos=np.searchsorted(rsf.unique_times_,180,side='right')-1
    risk_rsf=np.zeros(len(X_te)) if pos<0 else 1-rsf.predict_survival_function(X_te,return_array=True)[:,pos]
    result={'Original Cox':pd.Series(np.asarray(risk_cox),index=test.index),
            'Stratified Cox':pd.Series(np.asarray(risk_strat),index=test.index),
            'Random Survival Forest':pd.Series(risk_rsf,index=test.index)}
    assert all(v.notna().all() and v.between(0,1).all() for v in result.values())
    return result


## Table 16: Ten-split performance stability

Seeds are fixed in the exported table. RSF rank frequency is descriptive; no inferential test treats the ten overlapping test sets as independent.


In [ ]:
seeds=[42,43,44,45,46,47,48,49,50,51]
rows=[];seed42_predictions=None
for seed in seeds:
    train,test=train_test_split(baseline,test_size=0.2,random_state=seed,stratify=baseline.event_180)
    risks=fit_predict(train,test)
    yt=test.event_180.astype(int)
    if seed==42: seed42_predictions=(yt.copy(),{k:v.copy() for k,v in risks.items()})
    for name,p in risks.items():
        rows.append(dict(Seed=seed,Model=name,Test_N=len(yt),Deaths=int(yt.sum()),
             ROC_AUC=roc_auc_score(yt,p),Brier=brier_score_loss(yt,p)))
    print('Completed split',seed)
t16=pd.DataFrame(rows)
t16.to_csv(OUT/'Table16_Repeated_Split_Performance.csv',index=False)
display(t16.round(5))

## Table 17 and Figure 18: Paired differences and ranking frequency

In [ ]:
auc=t16.pivot(index='Seed',columns='Model',values='ROC_AUC')
brier=t16.pivot(index='Seed',columns='Model',values='Brier')
rows=[]
for other in ['Original Cox','Stratified Cox']:
    for metric,frame in [('ROC_AUC',auc),('Brier',brier)]:
        d=frame['Random Survival Forest']-frame[other]
        rows.append({'Contrast':f'RSF − {other}','Metric':metric,'Mean_difference':d.mean(),
                     'SD_across_splits':d.std(ddof=1),'Min_difference':d.min(),'Max_difference':d.max(),
                     'RSF_better_splits':int((d>0).sum() if metric=='ROC_AUC' else (d<0).sum()),'N_splits':len(d)})
t17=pd.DataFrame(rows)
winners=pd.DataFrame({'AUC_best':auc.idxmax(axis=1),'Brier_best':brier.idxmin(axis=1)},index=auc.index)
t17.to_csv(OUT/'Table17_Repeated_Split_Differences.csv',index=False)
winners.to_csv(OUT/'Table17b_Model_Ranking_Frequency.csv')
display(t17.round(5));display(winners.apply(pd.Series.value_counts).fillna(0).astype(int))
fig,axes=plt.subplots(1,2,figsize=(12,5))
for name,group in t16.groupby('Model'):
    axes[0].plot(group.Seed,group.ROC_AUC,marker='o',label=name)
    axes[1].plot(group.Seed,group.Brier,marker='o',label=name)
for ax,title in zip(axes,['ROC-AUC (higher is better)','Brier (lower is better)']):
    ax.set(xlabel='Train/test split seed',title=title);ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.suptitle('Figure 18. Repeated split sensitivity (10 stratified 80/20 splits)')
fig.tight_layout();fig.savefig(OUT/'Figure18_Repeated_Split_Performance.png',dpi=200)
plt.show()

### Calculation verification

The following check tests the fixed-risk-group formula on a small synthetic example. It ensures resampling preserves each patient's original risk group and that the weighted absolute group error equals the directly calculated grouped mean residual. **This is a software test, not a SUPPORT2 analysis result.**

In [ ]:
# Deterministic test; does not use clinical data or affect the bootstrap RNG.
_test_y = np.array([0, 1, 0, 1, 1, 0])
_test_p = np.array([.1, .2, .3, .7, .8, .9])
_test_g = np.array([0, 0, 0, 1, 1, 1])
_ix = np.array([0, 0, 2, 3, 4, 4])
def _group_error(y,p,g):
    return float(np.abs(np.bincount(g, weights=y-p, minlength=2)).sum()/len(y))
from pandas import DataFrame
_t = DataFrame({'y':_test_y[_ix],'p':_test_p[_ix],'g':_test_g[_ix]})
_a = _group_error(_test_y[_ix],_test_p[_ix],_test_g[_ix])
_b = float((_t.groupby('g').apply(lambda z: abs((z.y-z.p).sum()), include_groups=False)).sum()/len(_t))
assert np.isclose(_a,_b), (_a,_b)
assert np.array_equal(_test_g[_ix],np.array([0,0,0,1,1,1]))
print('Fixed-group calculation unit test passed.')

## Table 18: Paired calibration bootstrap with **fixed test-set risk groups**

The earlier implementation assigned new `qcut` deciles in each resampled dataset. Here we first calculate **ten model-specific risk-decile cutoffs** from the original seed-42 testing patients (the prediction probabilities). Each bootstrap sample reuses those *unchanged* cutoffs. This means a patient remains in the same risk group across bootstrap samples, and empty bootstrap groups simply contribute zero weight. All models use the same bootstrap row indices so model differences are paired.

**Outcomes:** Logistic calibration intercept and slope, predicted-minus-observed event-rate bias, and weighted mean absolute risk-group calibration error. The free-intercept logistic fit is *not* calibration-in-the-large in the offset-model sense. Bootstrap intervals describe conditional test-set uncertainty only. Because the last metric uses absolute deviations, its percentile intervals can be shifted upward and should be treated as exploratory rather than definitive proof of better calibration.


In [ ]:
from scipy.special import logit
import statsmodels.api as sm

B = 2000
rng = np.random.default_rng(20261107)
yt, models = seed42_predictions
Y = yt.to_numpy(dtype=int)
P = {name: values.reindex(yt.index).to_numpy(dtype=float) for name, values in models.items()}
assert len(Y) == 1821 and set(np.unique(Y)) == {0, 1}

# Cutoffs from original test-set predictions; fixed during *all* bootstrap resamples.
# Quantiles of raw risk (not reranked within resamples); -inf/+inf guarantee assignment.
def make_fixed_deciles(p):
    quantiles = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    edges = np.r_[-np.inf, quantiles, np.inf]
    if not np.all(np.diff(edges) > 0):
        raise ValueError('Nonunique risk-decile cutoffs: use a prespecified tie-aware strategy.')
    groups = np.searchsorted(quantiles, p, side='right')
    counts = np.bincount(groups, minlength=10)
    assert len(counts) == 10 and np.all(counts > 0), counts
    return edges, groups

fixed = {name: make_fixed_deciles(p) for name, p in P.items()}
cutoff_rows=[]
for name,(edges,grps) in fixed.items():
    for group in range(10):
        cutoff_rows.append({'Model':name,'Risk_group':group+1,
            'Lower_edge':edges[group],'Upper_edge':edges[group+1],
            'Original_test_n':int(np.sum(grps==group))})
pd.DataFrame(cutoff_rows).to_csv(OUT/'Table18b_Fixed_Risk_Group_Cutoffs.csv', index=False)

def calibrate_fixed(y, p, groups):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1-1e-6)
    groups = np.asarray(groups, dtype=int)
    assert len(y) == len(p) == len(groups)
    # Weighted group mean absolute error = (1/n) sum_g |sum_{i in g}(y_i-p_i)|.
    # Keeps the original grouping unchanged after re-sampling.
    residual_sums = np.bincount(groups, weights=y-p, minlength=10)
    decile_error = float(np.abs(residual_sums).sum() / len(y))
    design = sm.add_constant(logit(p), has_constant='add')
    fitted = sm.GLM(y, design, family=sm.families.Binomial()).fit(maxiter=100, disp=0)
    return {'Pred_minus_observed': float(np.mean(p)-np.mean(y)),
            'Free_calibration_intercept': float(fitted.params[0]),
            'Calibration_slope': float(fitted.params[1]),
            'Fixed_group_absolute_error': decile_error}

point = {name: calibrate_fixed(Y, p, fixed[name][1]) for name,p in P.items()}
boot = {name: [] for name in P}
accepted=0
for i in range(B):
    ix = rng.integers(0, len(Y), len(Y))
    if np.unique(Y[ix]).size < 2:
        continue
    for name, p in P.items():
        boot[name].append(calibrate_fixed(Y[ix], p[ix], fixed[name][1][ix]))
    accepted += 1
assert accepted == B, (accepted, B)

rows = []
for name in P:
    for metric, val in point[name].items():
        values = np.array([r[metric] for r in boot[name]], dtype=float)
        rows.append({'Model':name,'Metric':metric,'Point_estimate':val,
                     'Bootstrap_mean':float(values.mean()),
                     'Bootstrap_median':float(np.median(values)),
                     'CI_low':float(np.percentile(values,2.5)),
                     'CI_high':float(np.percentile(values,97.5)),
                     'Boot_N':len(values)})
t18 = pd.DataFrame(rows)
t18.to_csv(OUT/'Table18_Fixed_Group_Calibration_Bootstrap.csv', index=False)
display(t18.round(5))
print('All fixed-group boundaries were calculated once from the original test predictions.')
print('Bootstrap calibration summaries are conditional; absolute-error percentile intervals may be shifted.')


## Table 19: Paired fixed-group calibration contrasts

The original test-set estimate of each RSF-minus-Cox contrast and its percentile bootstrap interval are reported separately. All models use the same resampled patients; **each model retains its original risk-group cutoffs**. Intervals that include zero do not demonstrate statistically distinguishable contrasts; intervals excluding zero do not establish clinical importance. The absolute-error intervals may exhibit upward bias and should be interpreted with caution.


In [ ]:
comparison=[]
for other in ['Original Cox','Stratified Cox']:
    for metric in point['Random Survival Forest']:
        observed = point['Random Survival Forest'][metric] - point[other][metric]
        diffs=np.array([a[metric]-b[metric]
                        for a,b in zip(boot['Random Survival Forest'],boot[other])],dtype=float)
        low,high=np.percentile(diffs,[2.5,97.5])
        comparison.append({'Contrast':f'RSF − {other}', 'Metric':metric,
                           'Point_difference':observed, 'Bootstrap_mean_difference':float(diffs.mean()),
                           'CI_low':low, 'CI_high':high, 'CI_excludes_zero':bool(low>0 or high<0),
                           'Boot_N':len(diffs)})
t19=pd.DataFrame(comparison)
t19.to_csv(OUT/'Table19_Fixed_Group_Paired_Calibration_Differences.csv', index=False)
display(t19.round(5))


## Figure 19: Fixed-group calibration uncertainty

The dot represents the original test-set fixed-group calibration error and the line represents its conditional 95% percentile interval. The dot may lie outside the interval because the metric is non-smooth and can have bootstrap bias. Inference of superior calibration should not rest solely on these absolute-error intervals.


In [ ]:
d=t18[t18.Metric=='Fixed_group_absolute_error'].copy().reset_index(drop=True)
fig,ax=plt.subplots(figsize=(9,5))
for y,row in d.iterrows():
    low,high,estimate=(float(row[k]) for k in ['CI_low','CI_high','Point_estimate'])
    assert np.isfinite([low,high,estimate]).all() and low <= high, 'Invalid bootstrap summary'
    ax.hlines(y,low,high,color='tab:blue',linewidth=2)
    ax.plot([low,high],[y,y],marker='|',linestyle='None',markersize=12,color='tab:blue')
    ax.scatter(estimate,y,color='tab:orange',marker='o',zorder=3)
ax.set_yticks(range(len(d)),d['Model'])
ax.invert_yaxis()
ax.set_xlabel('Weighted mean absolute fixed-group calibration error')
ax.set_title('Figure 19. Fixed-group calibration: 2,000 paired bootstraps')
ax.grid(axis='x',alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'Figure19_Fixed_Group_Calibration_Bootstrap.png',dpi=200,bbox_inches='tight')
plt.show()
print('Line: conditional bootstrap percentile interval; dot: original estimate.')


## Validation, audit warnings and archive

Inspect the seed-42 performance reproduction, fixed-risk-group population sizes, and the calibration output before using the new results in the manuscript. This version does **not** assert that bootstrap intervals for absolute error are unbiased or that RSF calibration is significantly superior. The earlier Table 18/19 values and the new fixed-group values use different grouping procedures and must not be mixed.


In [ ]:
print('SEED 42 original comparison (compare with previously published notebook exports):')
display(t16[t16.Seed==42].round(4))
expected={'Original Cox':(.7124,.2156),'Stratified Cox':(.7099,.2164),'Random Survival Forest':(.7188,.2136)}
for name,(au,br) in expected.items():
    row=t16.query('Seed==42 and Model==@name').iloc[0]
    if abs(row.ROC_AUC-au)>.003 or abs(row.Brier-br)>.003:
        raise ValueError(f'Original seed-42 reproduction requires investigation: {name} {row.ROC_AUC:.4f}, {row.Brier:.4f}')
print('SEED-42 CHECK PASSED within predeclared ±0.003 tolerance; inspect full precision.')
import shutil
archive=shutil.make_archive('support2_new_results_fixed_groups','zip',root_dir=OUT)
print('Download outputs:',archive)